## File traces5000.bin
contains 5.000 power traces
Each power trace consists of 2.000 samples. The file therefore contains 2,000 x 5,000 power samples.
Every power sample is a signed 16-bit integer (int16_t, signed short). The file is therefore 2 x 2,000 x 5,000 = 20,000,000 bytes.

## File ciphertext5000.bin
contains 5.000 ciphertexts. Each ciphertext consists of 16 bytes. The file therefore contains 16 x 5,000 bytes.

little-endian

In every file, first, there are 2,000 samples (or 16 bytes of ciphertext respectively) collected during the first encryption. After that, there are 2,000 samples (or 16 bytes of ciphertext respectively) collected during the second encryption. And so on.

In [ ]:
from pathlib import Path
import numpy as np
# 5000 encryptions
# each 2000 traces
# each 16 Byte ciphertexts

data_dir: Path = Path.cwd() / 'ni-hsc-lab-02-data'
ptrace_length = 2000 # of 16 bit samples

ptraces = np.fromfile(data_dir / 'traces5000.bin', dtype=np.int16).reshape(-1, ptrace_length)
print(f'Power traces shape: {ptraces.shape}')

ct_length = 16 # Bytes
# load ciphertexts as unsigned bytes so values are in 0..255 (avoids negative indices when indexing SBoxInverse)
ciphertexts = np.fromfile(data_dir / 'ciphertext5000.bin', dtype=np.uint8).reshape(-1, ct_length)
print(f'Ciphertext shape: {ciphertexts.shape}')

assert ptraces.shape[0] == ciphertexts.shape[0]

# 2. Perform DPA attack (5 b)

Perform DPA and multi-bit DPA attacks on a single byte of key using the prepared data.

## Single-bit DPA

1. Choose a byte of key (subkey) to attack on. -> 1st byte

2. Choose following intermediate values:
- v1: result of the last but one round
- v2: ciphertext
- leakage function: L̂(v1,v2)=LSB(v1⊕v2).

3. Compute intermediate values for each of possible subkey values.
4. Divide power traces into two sets according to the leakage function.
5. Choose the most probable key candidate based on differences of means (using, e.g., maximum of DoM values).

6. Verify your choice using a graph displaying traces of differences of means for all of the key candidates highlighting the one you have chosen.


In [ ]:
ShiftRow = np.array([ 0, 5, 10, 15, 4, 9, 14, 3, 8, 13, 2, 7, 12, 1, 6, 11 ], dtype=np.uint8)
ShiftRowInverse = np.array([
    0, 13, 10, 7,   # row 0 → no shift
    4, 1, 14, 11,   # row 1 → shift right by 1
    8, 5, 2, 15,    # row 2 → shift right by 2
    12, 9, 6, 3     # row 3 → shift right by 3
], dtype=np.uint8)

SBoxInverse = np.array(
            [0x52 ,0x09 ,0x6A ,0xD5 ,0x30 ,0x36 ,0xA5 ,0x38 ,0xBF ,0x40 ,0xA3 ,0x9E ,0x81 ,0xF3 ,0xD7 ,0xFB
            ,0x7C ,0xE3 ,0x39 ,0x82 ,0x9B ,0x2F ,0xFF ,0x87 ,0x34 ,0x8E ,0x43 ,0x44 ,0xC4 ,0xDE ,0xE9 ,0xCB
            ,0x54 ,0x7B ,0x94 ,0x32 ,0xA6 ,0xC2 ,0x23 ,0x3D ,0xEE ,0x4C ,0x95 ,0x0B ,0x42 ,0xFA ,0xC3 ,0x4E
            ,0x08 ,0x2E ,0xA1 ,0x66 ,0x28 ,0xD9 ,0x24 ,0xB2 ,0x76 ,0x5B ,0xA2 ,0x49 ,0x6D ,0x8B ,0xD1 ,0x25
            ,0x72 ,0xF8 ,0xF6 ,0x64 ,0x86 ,0x68 ,0x98 ,0x16 ,0xD4 ,0xA4 ,0x5C ,0xCC ,0x5D ,0x65 ,0xB6 ,0x92
            ,0x6C ,0x70 ,0x48 ,0x50 ,0xFD ,0xED ,0xB9 ,0xDA ,0x5E ,0x15 ,0x46 ,0x57 ,0xA7 ,0x8D ,0x9D ,0x84
            ,0x90 ,0xD8 ,0xAB ,0x00 ,0x8C ,0xBC ,0xD3 ,0x0A ,0xF7 ,0xE4 ,0x58 ,0x05 ,0xB8 ,0xB3 ,0x45 ,0x06
            ,0xD0 ,0x2C ,0x1E ,0x8F ,0xCA ,0x3F ,0x0F ,0x02 ,0xC1 ,0xAF ,0xBD ,0x03 ,0x01 ,0x13 ,0x8A ,0x6B
            ,0x3A ,0x91 ,0x11 ,0x41 ,0x4F ,0x67 ,0xDC ,0xEA ,0x97 ,0xF2 ,0xCF ,0xCE ,0xF0 ,0xB4 ,0xE6 ,0x73
            ,0x96 ,0xAC ,0x74 ,0x22 ,0xE7 ,0xAD ,0x35 ,0x85 ,0xE2 ,0xF9 ,0x37 ,0xE8 ,0x1C ,0x75 ,0xDF ,0x6E
            ,0x47 ,0xF1 ,0x1A ,0x71 ,0x1D ,0x29 ,0xC5 ,0x89 ,0x6F ,0xB7 ,0x62 ,0x0E ,0xAA ,0x18 ,0xBE ,0x1B
            ,0xFC ,0x56 ,0x3E ,0x4B ,0xC6 ,0xD2 ,0x79 ,0x20 ,0x9A ,0xDB ,0xC0 ,0xFE ,0x78 ,0xCD ,0x5A ,0xF4
            ,0x1F ,0xDD ,0xA8 ,0x33 ,0x88 ,0x07 ,0xC7 ,0x31 ,0xB1 ,0x12 ,0x10 ,0x59 ,0x27 ,0x80 ,0xEC ,0x5F
            ,0x60 ,0x51 ,0x7F ,0xA9 ,0x19 ,0xB5 ,0x4A ,0x0D ,0x2D ,0xE5 ,0x7A ,0x9F ,0x93 ,0xC9 ,0x9C ,0xEF
            ,0xA0 ,0xE0 ,0x3B ,0x4D ,0xAE ,0x2A ,0xF5 ,0xB0 ,0xC8 ,0xEB ,0xBB ,0x3C ,0x83 ,0x53 ,0x99 ,0x61
            ,0x17 ,0x2B ,0x04 ,0x7E ,0xBA ,0x77 ,0xD6 ,0x26 ,0xE1 ,0x69 ,0x14 ,0x63 ,0x55 ,0x21 ,0x0C ,0x7D],
            dtype=np.uint8)

In [ ]:
def calc_intermediate_values(ct, key_guess, ct_byte_idx):
    """
    :param ct: A single ciphertext.
    :param key_guess: Guess of a single key byte.
    :param ct_byte_idx: Attacked byte indexof the ciphertext.
    :return: Hypothesis of state register after 9. round and state register after 10. round.
    """
    post_round10 = ct[ct_byte_idx]

    # States after given operation. The 10th round in reverse.
    # SR shifts bytes of states, therefore perform the reverse to attack correct corresponding bytes.
    # Ex.: Byte in state register on index 1 after 10th round was byte on index 13 after 9th round.
    post_sub_bytes = ct[ShiftRowInverse[ct_byte_idx]] ^ key_guess
    post_round9 = SBoxInverse[post_sub_bytes]
    return post_round10, post_round9

In [ ]:
def calc_leakage_function(v1, v2):
    """
    Extract LSB from XOR of intermediate values.
    """
    return 0x0001 & (v1 ^ v2)

In [ ]:
def compute_doms(doms, dom_time_matrix, key_guess, l0, l1):
    """
    Differentiate leakage groups using Difference of Means and return key candidate.
    :param doms: maximal differences of means (of leakage groups) within their corresponding time series'
    :param dom_time_matrix: doms time series for each key guess
    :param l0: traces from leakage group with LSB==0
    :param l1: traces from leakage group with LSB==1
    """
    # compute per-sample mean for each group; handle empty groups safely
    if len(l0) > 0: mean0 = np.mean(np.vstack(l0), axis=0)
    else: mean0 = np.zeros(ptraces.shape[1], dtype=ptraces.dtype)

    if len(l1) > 0: mean1 = np.mean(np.vstack(l1), axis=0)
    else: mean1 = np.zeros(ptraces.shape[1], dtype=ptraces.dtype)

    dom = np.abs(mean0 - mean1)
    dom_time_matrix[key_guess] = dom
    doms[key_guess] = dom.max()



In [ ]:
# Perform single-bit DPA

# attacked byte index
key_byte_idx = 0

n_keys = 256
n_samples = ptraces.shape[1]
# container for per-key difference-of-means time series
dom_time_matrix = np.zeros((n_keys, n_samples), dtype=np.float32)
doms = np.zeros(n_keys, dtype=np.float32)   # scalar metric per key

for key_guess in range(n_keys):
    # full traces per leakage group (we need per-sample means)
    lgroup0_traces = []
    lgroup1_traces = []

    for ptrace, ct in zip(ptraces, ciphertexts):
        v1, v2 = calc_intermediate_values(ct, key_guess, key_byte_idx)
        L = calc_leakage_function(v1, v2)
        (lgroup0_traces if L == 0 else lgroup1_traces).append(ptrace)

    compute_doms(doms, dom_time_matrix, key_guess, lgroup0_traces, lgroup1_traces)

key_candidate = np.argmax(doms)
print(f'Round 10 roundkey candidate: {hex(key_candidate)}')

In [ ]:
import matplotlib.pyplot as plt
# Plot dom, highlight chosen candidate
fig, ax = plt.subplots(figsize=(12, 6))
for k in range(n_keys):
    ax.plot(dom_time_matrix[k], color='gray', alpha=0.25, linewidth=0.6)

ax.plot(dom_time_matrix[key_candidate], color='red', linewidth=2.2, label=f'chosen: {hex(key_candidate)}')
ax.set_xlabel('Sample index')
ax.set_ylabel('Absolute difference of means')
ax.set_title('Per-sample difference-of-means for all key candidates')
ax.legend()
ax.grid(alpha=0.3)
plt.show()

Follow the procedure of single-bit DPA on the same key byte, but repeat the procedure for each of 8 bits of function L̂.

Sum up the means for every bit and evaluate the best key candidate.

Display differences of means for the best candidate for Single-bit and Multi-bit variant in a graph.

Repeat the whole procedure decreasing the number of power traces used to verify that a lower amount of data is needed for the Multi-bit variant.

In [ ]:
def calc_leakage_function_perbit(v1, v2, bit):
    """Return 0/1 for the given bit of (v1 ^ v2)."""
    return ((int(v1) ^ int(v2)) >> bit) & 1


In [ ]:
# Perform multi-bit DPA
# attacked byte index
key_byte_idx = 0

n_keys = 256
n_samples = ptraces.shape[1]
# container for per-key difference-of-means time series
dom_time_matrix = np.zeros((n_keys, n_samples), dtype=np.float32)
doms = np.zeros(n_keys, dtype=np.float32)   # scalar metric per key

for key_guess in range(n_keys):
    dom_time_sum = np.zeros(n_samples, dtype=np.float32)
    for bit in range(8):
        lgroup0_traces = []
        lgroup1_traces = []
        for ptrace, ct in zip(ptraces, ciphertexts):
            v1, v2 = calc_intermediate_values(ct, key_guess, key_byte_idx)
            L = calc_leakage_function_perbit(v1, v2, bit)
            (lgroup0_traces if L == 0 else lgroup1_traces).append(ptrace)

        if lgroup0_traces: mean0 = np.mean(np.vstack(lgroup0_traces), axis=0)
        else: mean0 = np.zeros(n_samples, dtype=ptraces.dtype)

        if lgroup1_traces: mean1 = np.mean(np.vstack(lgroup1_traces), axis=0)
        else: mean1 = np.zeros(n_samples, dtype=ptraces.dtype)

        dom_bit = np.abs(mean0 - mean1)
        dom_time_sum += dom_bit

    dom_time_matrix[key_guess] = dom_time_sum
    doms[key_guess] = dom_time_sum.max()

key_candidate = np.argmax(doms)
print(f'Round 10 roundkey candidate (multi-bit): {hex(key_candidate)}')

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
for k in range(n_keys):
    ax.plot(dom_time_matrix[k], color='gray', alpha=0.25, linewidth=0.6)

ax.plot(dom_time_matrix[key_candidate], color='red', linewidth=2.2, label=f'chosen: {hex(key_candidate)}')
ax.set_xlabel('Sample index')
ax.set_ylabel('Summed absolute difference of means')
ax.set_title('Multi-bit DPA: summed per-bit difference-of-means for all key candidates')
ax.legend()
ax.grid(alpha=0.3)
plt.show()